In [ ]:
import requests
import pandas as pd
import time
import datetime
import os
from dotenv import load_dotenv

# --- 1. Identity & Privacy ---
load_dotenv("../../config/.env") 
contact_email = os.getenv("CONTACT_EMAIL", "anonymous@example.com")

# --- 2. Notebook-Specific Configuration ---
SOURCE_PREFIX = "SNOMED"
SOURCE_NAME = "SNOMED CT"
BASE_URI = "http://snomed.info/id/"

# Project-wide file naming standardized here
PREFIX_MAP_FILE = "../../config/prefix_map.csv"
raw_ingest_file = "../../data/raw/raw_snomed.csv"
master_file = "../../data/raw/metadata_ingest_undeduped.csv"

# Headers for polite API requests
HEADERS = {
    'User-Agent': f'ReligiousMappingProject/1.0 (Research; mailto:{contact_email})'
}

# --- 3. Prefix Map Management ---
def update_prefix_map(prefix, base_uri, source_name):
    """Maintains a master list of prefixes and their base URIs for the project."""
    new_entry = pd.DataFrame([{
        'Prefix': prefix, 
        'Base_URI': base_uri, 
        'Source_Name': source_name
    }])
    
    if os.path.exists(PREFIX_MAP_FILE):
        pmap = pd.read_csv(PREFIX_MAP_FILE)
        if prefix not in pmap['Prefix'].values:
            pmap = pd.concat([pmap, new_entry], ignore_index=True)
            pmap.to_csv(PREFIX_MAP_FILE, index=False)
    else:
        new_entry.to_csv(PREFIX_MAP_FILE, index=False)
    print(f"--- Prefix Map verified for {prefix} ---")

update_prefix_map(SOURCE_PREFIX, BASE_URI, SOURCE_NAME)

# --- 4. Request Handling (Updated with 3-try retry) ---
def safe_request(url, params=None):
    """API request wrapper with error handling and retry loop."""
    for attempt in range(3):
        try:
            response = requests.get(url, params=params, headers=HEADERS, timeout=15)
            if response.status_code == 200:
                return response.json()
            elif response.status_code == 404: # Don't retry a hard 404 Not Found
                return None
            print(f"\n[!] API Error {response.status_code} on attempt {attempt+1}: {response.reason}")
            time.sleep(2)
        except Exception as e:
            print(f"\n[!] Connection Error on attempt {attempt+1}: {e}")
            time.sleep(2)
    return None

# --- 5. Hierarchy Logic with Safety Flags ---
path_cache = {}

def get_full_path(sctid, depth=0):
    """
    Recursively traces parents to build breadcrumbs. 
    Returns a warning flag if the depth limit is exceeded.
    """
    sctid = str(sctid)
    if sctid in path_cache: 
        return path_cache[sctid]
    
    # Safety check: prevents infinite loops and massive API usage
    if depth > 10:
        return "!!! DEPTH LIMIT REACHED !!!"
        
    if sctid == '138875005': # SNOMED Root Concept
        return ""
    
    browser_url = f"https://snowstorm.ihtsdotools.org/snowstorm/snomed-ct/browser/MAIN/concepts/{sctid}"
    res = safe_request(browser_url)
    if not res: return sctid
        
    term = res.get('pt', {}).get('term', sctid)
    # Get active 'Is a' (parent) relationships
    parents = [r['destinationId'] for r in res.get('relationships', []) 
               if r['active'] and r['typeId'] == '116680003']
    
    if not parents:
        path_cache[sctid] = term
        return term
    
    # Recurse to find the next parent up
    parent_path = get_full_path(parents[0], depth + 1)
    
    # If the parent returned a warning, pass it down to the child
    if "DEPTH LIMIT REACHED" in parent_path:
        return "!!! DEPTH LIMIT REACHED !!!"

    full_p = f"{parent_path} > {term}" if parent_path else term
    path_cache[sctid] = full_p
    return full_p

print("Cell 1: Environment and functions initialized.")

In [ ]:
# 1. Automated Research Seeds
SEED_CONCEPTS = [
    "105390002",  # Spiritual therapy / Spiritual care
    "11015003",   # Minister of religion / clergy
    "1400009",    # Religion and philosophy (Root)
    "183945002",  # Procedure declined for religious reason
    "410597007",  # Person by affiliation with belief system
    "408892004"   # Finding of religious/philosophical belief
]

all_processed_ids = set()

# Header order
COLUMN_ORDER = [
    "Source_System", "Primary_Label", "CURIE", "Formal_Label", 
    "Hierarchy_Path", "Synonyms", "Description", "Parent_IDs", 
    "Concept_ID", "URI", "Status", "Extraction_Date"
]

# Clean up raw file before starting
if os.path.exists(raw_ingest_file): os.remove(raw_ingest_file)

timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

for seed_id in SEED_CONCEPTS:
    print(f"\n--- Starting Harvest for Seed: {seed_id} ---")
    
    fhir_url = "https://snowstorm.ihtsdotools.org/fhir/ValueSet/$expand"
    params = {
        'url': f'http://snomed.info/sct?fhir_vs=ecl/<<{seed_id}',
        'count': 1000
    }

    data = safe_request(fhir_url, params)

    if not data or 'expansion' not in data:
        print(f"Expansion failed for {seed_id}. Skipping.")
        continue

    items = data.get('expansion', {}).get('contains', [])
    initial_list = [{'Concept_ID': str(item['code']), 'Primary_Label': item.get('display')} for item in items]
    total = len(initial_list)
    
    print(f"Found {total} concepts. Enriching and appending to CSV...")

    browser_base = "https://snowstorm.ihtsdotools.org/snowstorm/snomed-ct/browser/MAIN/concepts"
    
    for i, entry in enumerate(initial_list, 1):
        cid = entry['Concept_ID']
        
        if cid in all_processed_ids:
            continue
            
        status_text = f"[{i}/{total}] Processing: {entry['Primary_Label']} ({cid})"
        print(f"\r{status_text:<120}", end="")
        
        res = safe_request(f"{browser_base}/{cid}")
        
        if res:
            # Metadata extraction
            syns = [d['term'] for d in res.get('descriptions', []) if d['active'] and d['type'] == 'SYNONYM']
            parents = [r['destinationId'] for r in res.get('relationships', []) if r['active'] and r['typeId'] == '116680003']
            defs = [d['term'] for d in res.get('descriptions', []) if d['active'] and d['type'] == 'TEXT_DEFINITION']
            fsn = res.get('fsn', {}).get('term', '')
            
            # Objective status check
            active_status = "active" if res.get('active') else "inactive"

            hierarchy = get_full_path(cid)
            
            row = {
                'Source_System': SOURCE_NAME,
                'Primary_Label': res.get('pt', {}).get('term', entry['Primary_Label']),
                'CURIE': f"{SOURCE_PREFIX}:{cid}",
                'Formal_Label': fsn,
                'Hierarchy_Path': hierarchy,
                'Synonyms': " | ".join(syns) if syns else "",
                'Description': " ".join(defs) if defs else "", 
                'Parent_IDs': " | ".join([str(p) for p in parents]) if parents else "",
                'Concept_ID': str(cid),
                'URI': f"{BASE_URI}{cid}",
                'Status': active_status, 
                'Extraction_Date': timestamp
            }
            
            df_row = pd.DataFrame([row])[COLUMN_ORDER] 
            file_exists = os.path.isfile(raw_ingest_file)
            df_row.to_csv(raw_ingest_file, mode='a', index=False, header=not file_exists, encoding='utf-8-sig')
            
            all_processed_ids.add(cid)
        else:
            print(f"\n[!] Skipping {cid} (API returned no data).")

        time.sleep(1.2)

print(f"\n\nDone! Total unique concepts processed: {len(all_processed_ids)}")
print(f"Data stored in {raw_ingest_file}.")

In [ ]:
if os.path.exists(raw_ingest_file):
    df_new = pd.read_csv(raw_ingest_file)
    
    if os.path.exists(master_file):
        df_master = pd.read_csv(master_file)
        
        # Deduplication Check based on URI
        existing_uris = set(df_master['URI'].dropna().unique())
        df_to_add = df_new[~df_new['URI'].isin(existing_uris)]
        
        if not df_to_add.empty:
            df_final = pd.concat([df_master, df_to_add], ignore_index=True)
            df_final.to_csv(master_file, index=False, encoding='utf-8-sig')
            print(f"Added {len(df_to_add)} new {SOURCE_PREFIX} records to master.")
        else:
            print(f"No new {SOURCE_PREFIX} records found to add.")
    else:
        df_new.to_csv(master_file, index=False, encoding='utf-8-sig')
        print(f"Master file created with {len(df_new)} {SOURCE_PREFIX} records.")
else:
    print(f"Error: {raw_ingest_file} not found. Please run the ingest sequence first.")